# 02 — Auditoria
Análise de completude, unicidade, datas e consistência das contagens.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
for p in [DATA/"raw", DATA/"processed", OUT/"auditoria", OUT/"eda", OUT/"estatistica", OUT/"modelos", OUT/"interpretabilidade", OUT/"modelo_final"]:
    p.mkdir(parents=True,exist_ok=True)
SILVER = DATA/"processed"/"prf_acidentes_silver_2020_2025.parquet"
df=pd.read_parquet(DATA/"processed"/"prf_acidentes_2020_2025.parquet")
audit=pd.DataFrame({"coluna":df.columns,"nulos":[df[c].isna().sum() for c in df],"pct_nulos":[df[c].isna().mean()*100 for c in df],"distintos":[df[c].nunique(dropna=True) for c in df]})
display(audit.sort_values("nulos",ascending=False))
audit.to_csv(OUT/"auditoria"/"auditoria_qualidade_prf.csv",index=False)
print("IDs repetidos:",df.id.duplicated().sum())
dt=pd.to_datetime(df.data_inversa,errors="coerce")
print("Datas inválidas:",dt.isna().sum())
print("Ano discordante:",((dt.dt.year!=df.ano_fonte)&dt.notna()).sum())

In [ ]:
for c in ["pessoas","mortos","feridos_leves","feridos_graves","ilesos","ignorados","feridos"]:
    df[c]=pd.to_numeric(df[c],errors="coerce")
print("feridos diferente de leves+graves:",(df.feridos!=df.feridos_leves+df.feridos_graves).sum())
soma=df[["mortos","feridos_leves","feridos_graves","ilesos","ignorados"]].sum(axis=1)
print("pessoas diferente de soma dos estados:",(df.pessoas!=soma).sum())
print("Combinações de traçado:",df.tracado_via.nunique())